In [11]:
import jupedsim as jps 
from shapely import Polygon
import pathlib

trajectory_file = pathlib.Path("simple_evac.sqlite")
if trajectory_file.exists():
    trajectory_file.unlink()
room = Polygon([(0, 0), (10, 0), (10, 8), (0, 8)])

simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),
    geometry=room,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=pathlib.Path("simple_evac.sqlite"),
        every_nth_frame=5
    )
)
exit_right = Polygon([
    (9.5, 3),
    (10, 3),
    (10, 5),
    (9.5, 5)
])
exit_left = Polygon([
    (0, 3),
    (0.5, 3),
    (0.5, 5),
    (0, 5)
])

exit_id_right = simulation.add_exit_stage(exit_right)
exit_id_left = simulation.add_exit_stage(exit_left)

journey_left = jps.JourneyDescription([exit_id_left])
journey_right = jps.JourneyDescription([exit_id_right])
journey_id_left = simulation.add_journey(journey_left)
journey_id_right = simulation.add_journey(journey_right)

start_positions = [(x,y) for x in range(3,7) for y in range(3,7)]
for pos in start_positions:
    parameters = jps.CollisionFreeSpeedModelAgentParameters(position=pos,journey_id=journey_id_right,stage_id=exit_id_right)
    simulation.add_agent(parameters)

steps = 0
try:
    while simulation.agent_count() >0:
        simulation.iterate()
        steps+=1
finally:
    simulation._writer.close()
print(f"steps {steps}, agent count {simulation.agent_count()}")


RuntimeError: Unknown journey id: 2

In [8]:
from jupedsim.internal.notebook_utils import animate, read_sqlite_file
trajectory_data, walkable_area = read_sqlite_file("simple_evac.sqlite")

animation = animate(
    trajectory_data,
    walkable_area,
    every_nth_frame=2
)

animation.show()

In [9]:
import nbformat

print(nbformat.__version__)

5.11.1
